<a href="https://colab.research.google.com/github/rayvankaazrifany/skripsi-nlp/blob/main/kerkom_NLP_baru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
import pandas as pd
import numpy as np
import re # regex (regular expression)
import os
import string
import nltk # tokenizing, stopword, stemming, lemmatization
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory   # pip install PySastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory # Tambahkan ini

In [75]:
# Pastikan data pendukung NLTK terunduh
nltk.download('punkt') # tokenization (memecah teks)
nltk.download('stopwords') # stopword removal (hapus kata umum ga bermakna)
nltk.download('wordnet') # lemmatization ubah kata ke bentuk dasar kamus
nltk.download('omw-1.4') # omw open multilingual wordnet. database kosakata berisi hubungan antar kata (semantic relationship)
nltk.download('punkt_tab') # tabel aturan tambahan agar pemisahan kalimat (sentence tokenization) lebih akurat


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [76]:
# Load Data
df = pd.read_excel('dataset_google-maps-reviews-rs-cicendo-personal-cleaned_translated.xlsx')

In [77]:
# --- Sumber 1: kbba.txt (tab-separated, no header) ---
url_kbba = "https://raw.githubusercontent.com/ramaprakoso/analisis-sentimen/master/kamus/kbba.txt"
kamus_kbba = pd.read_csv(url_kbba, sep='\t', header=None)
dict_kbba = dict(zip(kamus_kbba[0], kamus_kbba[1]))

# --- Sumber 2: colloquial-indonesian-lexicon.csv ---
url_alay = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/refs/heads/master/colloquial-indonesian-lexicon.csv"
kamus_alay = pd.read_csv(url_alay)
dict_alay = dict(zip(kamus_alay['slang'], kamus_alay['formal']))

# --- Sumber 3: Custom dictionary ---
custom_dict = {
    'rs': 'rumah sakit',
    'rsud': 'rumah sakit umum daerah',
    'rsup': 'rumah sakit umum pusat',
    'igd': 'instalasi gawat darurat',
    'poli': 'poliklinik',
    'dr': 'dokter',
    'drg': 'dokter gigi',
    'sp': 'spesialis',
    'tht': 'telinga hidung tenggorokan',
    'rj': 'rawat jalan',
    'ri': 'rawat inap',
    'lbh': 'lebih',
    'th': 'tahun',
    'tk': 'tidak',
    'dibatu' : 'dibantu',
    'dilayanin' : 'dilayani',
    'terimakasih' : 'terima kasih',
    'makasih' : 'terima kasih',
    'nemenin' : 'menemani',
}

# --- Gabungkan ketiganya ---
# Urutan update penting: yang belakangan akan override yang sebelumnya
slang_dict = {}
slang_dict.update(dict_kbba)      # Sumber 1 (prioritas terendah)
slang_dict.update(dict_alay)      # Sumber 2
slang_dict.update(custom_dict)    # Sumber 3 (prioritas tertinggi)

print(f"Total entri slang_dict: {len(slang_dict)}")

Total entri slang_dict: 5257


In [78]:
def nlp_clean_safe(text):
    # 1. URL & Arbitrary String
    text = re.sub(r'https?://\s+|www\.\s+', '', text)
    text = re.sub(r'@[^\s]+', '', text)
    text = re.sub(r'#[^\s]+', '', text)
    text = re.sub(r'[^\x00-\x7f]', '', text)

    # 2. Case Folding
    text = text.lower()

    # 3. Slang Normalization
    text = " ".join([slang_dict.get(w, w) for w in text.split()])

    # 4. Segmentasi Kalimat
    sentences = sent_tokenize(text)

    final_results = []

    # 🔥 Stopword Sastrawi
    factory_stop = StopWordRemoverFactory()
    stop_words = set(factory_stop.get_stop_words())

    # (opsional) custom stopword
    custom_stopwords = {'yg', 'aja', 'deh'}
    stop_words = stop_words.union(custom_stopwords)

    # 🔥 Stemmer Sastrawi (lebih cocok Indo)
    factory_stem = StemmerFactory()
    stemmer = factory_stem.create_stemmer()

    # ❌ Lemmatizer NLTK kurang cocok Indo (tapi tetap boleh dipakai)
    lemmatizer = WordNetLemmatizer()

    for sent in sentences:
        # 5. Hapus tanda baca
        sent_no_punc = sent.translate(str.maketrans('', '', string.punctuation))

        # 6. Tokenisasi
        tokens = word_tokenize(sent_no_punc)

        # 7. Stopword Removal
        tokens_clean = [w for w in tokens if w not in stop_words]

        # 8. Stemming (Sastrawi)
        stemmed_sentence = stemmer.stem(" ".join(tokens_clean))
        stemmed = stemmed_sentence.split()

        # 9. Lemmatization (optional)
        lemmatized = [lemmatizer.lemmatize(w) for w in tokens_clean]

        final_results.append({
            'original': sent,
            'tokens': tokens_clean,
            'stemmed': stemmed,
            'lemmatized': lemmatized
        })

    return final_results

In [79]:
# Ambil 5 baris pertama
sample_texts = df['text'].head(5)

print("=== HASIL PREPROCESSING (5 DATA) ===\n")

for i, text in enumerate(sample_texts, 1):
    print(f"\n========== REVIEW {i} ==========")

    processed = nlp_clean_safe(text)

    for j, res in enumerate(processed, 1):
        print(f"\n--- Kalimat {j} ---")
        print(f"Original    : {res['original']}")
        print(f"Tokens      : {res['tokens']}")
        print(f"Stemmed     : {res['stemmed']}")
        print(f"Lemmatized  : {res['lemmatized']}")

=== HASIL PREPROCESSING (5 DATA) ===


========== REVIEW 1 ==========

--- Kalimat 1 ---
Original    : terima kasih rumah sakit mata cicendo!
Tokens      : ['terima', 'kasih', 'rumah', 'sakit', 'mata', 'cicendo']
Stemmed     : ['terima', 'kasih', 'rumah', 'sakit', 'mata', 'cicendo']
Lemmatized  : ['terima', 'kasih', 'rumah', 'sakit', 'mata', 'cicendo']

--- Kalimat 2 ---
Original    : pelayanannya baik staf nya ramah dan fasilitasnya memadai!
Tokens      : ['pelayanannya', 'baik', 'staf', 'nya', 'ramah', 'fasilitasnya', 'memadai']
Stemmed     : ['layan', 'baik', 'staf', 'nya', 'ramah', 'fasilitas', 'pada']
Lemmatized  : ['pelayanannya', 'baik', 'staf', 'nya', 'ramah', 'fasilitasnya', 'memadai']

========== REVIEW 2 ==========

--- Kalimat 1 ---
Original    : terima kasih sudah dibantu daftar
Tokens      : ['terima', 'kasih', 'dibantu', 'daftar']
Stemmed     : ['terima', 'kasih', 'bantu', 'daftar']
Lemmatized  : ['terima', 'kasih', 'dibantu', 'daftar']

========== REVIEW 3 ==========

-

In [80]:
import pandas as pd

# Function to apply nlp_clean_safe and consolidate results per review for export
def get_processed_features_for_export(text):
    processed_sentences = nlp_clean_safe(text)

    # Initialize lists to store aggregated results
    all_original_sentences = []
    all_tokens = []
    all_stemmed = []
    all_lemmatized = []

    for sentence_result in processed_sentences:
        all_original_sentences.append(sentence_result['original'])
        all_tokens.extend(sentence_result['tokens'])
        all_stemmed.extend(sentence_result['stemmed'])
        all_lemmatized.extend(sentence_result['lemmatized'])

    return {
        'segmented_sentences': ' | '.join(all_original_sentences), # Join original sentences
        'tokens': ' '.join(all_tokens),                            # Join all tokens
        'stemmed': ' '.join(all_stemmed),                          # Join all stemmed words
        'lemmatized': ' '.join(all_lemmatized)                     # Join all lemmatized words
    }

# Apply the function to the 'text' column
processed_data_for_export_series = df['text'].apply(get_processed_features_for_export)

# Convert the Series of dictionaries into a DataFrame with separate columns
processed_features_for_export_df = processed_data_for_export_series.apply(pd.Series)

# Concatenate the new feature columns to the original DataFrame
df_export = pd.concat([df, processed_features_for_export_df], axis=1)

# Export the final DataFrame to an Excel file
df_export.to_excel('review_rs_cicendo_preprocessed_full_features.xlsx', index=False)

print("Data exported successfully to 'review_rs_cicendo_preprocessed_full_features.xlsx'")

Data exported successfully to 'review_rs_cicendo_preprocessed_full_features.xlsx'
